# Прогноз и демонстрационная нейросеть

Все данные синтетические. Работа выполняется в отдельной ветке; эталон v2 не изменяется.

Найдём корень проекта. В Colab при отсутствии файлов клонируется только рабочая ветка; локально используются уже полученные исходники.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
while root != root.parent and not (root/'efc').exists():
    root = root.parent
if not (root/'efc').exists():
    import subprocess
    target = Path('/content/Forecast_Edge_Fog_Cloud')
    subprocess.run(['git','clone','--branch','night/fefc-v3','https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(target)],check=True)
    root=target
os.chdir(root)
sys.path.insert(0,str(root))
print('Рабочая папка:',root)


Что исследуем: один интерфейс для пяти прогнозов. Отдельный синтетический сценарий с четырьмя неравномерно нагруженными Fog не подменяет базовый сценарий парковки.

In [ ]:
from efc.forecast.study import forecast_study
from efc.forecast.views import plot_forecasts
from efc.forecast.models import FrozenNetwork,state_of,to_ranks,mm1k
import pandas as pd,json
metrics,predictions,models,sim=forecast_study()
display(metrics)

Входы обучения, калибровки и проверки разделены по времени завершения задач. На запрос доступны только ранее завершённые ответы; потерянные события не имеют целевой задержки и считаются отдельно.

In [ ]:
meta=json.loads(Path('results/forecast_metadata.json').read_text())
print(meta)
assert meta['train_end_max_s']<meta['cut_train_s']
assert meta['calibration_end_max_s']<meta['cut_test_s']
assert (predictions.last_observed_completion_s<predictions.query_s).all()

Схема показывает заменяемую «розетку». MAE и покрытие p95 — разные показатели: хорошая точечная ошибка не означает правильного хвоста; фактическое недопокрытие не скрывается.

In [ ]:
plot_forecasts(metrics,predictions)

Файл my_network.py — пример адаптера для вашей сети. Сохранённые веса демонстрационного MLP проверяем отдельным вычислением без pickle; порядковые команды не подставляются вместо секунд.

In [ ]:
frozen=FrozenNetwork(json.loads(Path('results/demo_network.json').read_text()))
s=state_of(next(r for r in sim['tasks'] if r['status']=='completed'))
a=frozen.predict(s);b=models['NeuralNet'].predict(s)
assert abs(a['mean_ms']-b['mean_ms'])<1e-7
print(a)
print('Порядковый пример:',to_ranks({'F0':a,'F1':{'mean_ms':a['mean_ms']+1,'p95_ms':a['p95_ms']+1}}))

Проверка: таблица для статьи читается из того же файла результатов. M/M/1/K — только приближённый baseline; равенство Литтла проверяется внутри его предпосылок.

In [ ]:
q=mm1k(10,.055,51)
assert abs(q['L']-q['lambda_eff']*q['W_s'])<1e-10
from_file=pd.read_csv('results/forecast_metrics.csv')
assert len(from_file)==5
assert ((from_file.coverage_p95>=0)&(from_file.coverage_p95<=1)).all()
display(from_file[['method','mae_ms','coverage_p95']].rename(columns={'mae_ms':'MAE в статье = модели','coverage_p95':'покрытие в статье = модели'}))
print('OK')